<a href="https://colab.research.google.com/github/guilhermelaviola/ApplicationsOfDataScienceInDisruptiveTechnologies/blob/main/Class16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Future Innovations in Disruptive Technologies**
Disruptive technological innovations, such as deep neural networks (CNNs and RNNs), the expansion of mobile and IoT devices (driven by 5G and 6G networks), and the emergence of quantum computing, are revolutionizing object tracking by enabling massive real-time data processing and precise prediction of trajectories and behaviors. These tools find fundamental practical applications in individual health monitoring, smart city security, and precision agriculture, transforming the operation of businesses and governments. However, this technological advancement brings proportional challenges related to privacy, data security, and the need for a highly skilled workforce, making a constant dialogue on regulation and ethics imperative to ensure that social and technological benefits are achieved in a balanced and safe manner.

In [1]:
# Importing all the necessary libraries and resources:
import torch
import torch.nn as nn
import numpy as np

## **Example: Tracker with Recurrent Neural Networks (RNNs)**
The following example shows how to use PyTorch to build a simple RNN that takes the past coordinates of an object (like a car or a drone) and predicts where it will move next.

In [2]:
# Defining the RNN Architecture for Trajectory Prediction:
class TrajectoryTrackerRNN(nn.Module):
    def __init__(self, input_size=2, hidden_size=16, output_size=2, num_layers=1):
        super(TrajectoryTrackerRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer to process the temporal sequence of coordinates:
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        # Fully connected layer to map hidden state to the next (x, y) coordinate:
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initializing hidden state with zeros:
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        # Forward pass through RNN:
        out, _ = self.rnn(x, h0)

        # We only care about the prediction at the very last time step:
        out = self.fc(out[:, -1, :])
        return out

In [3]:
# Generating Simulated Trajectory Data:
simulated_past_coordinates = np.array([
    [1.0, 1.2],
    [2.0, 2.1],
    [3.1, 2.9],
    [4.0, 4.2],
    [4.9, 5.0]
], dtype=np.float32)

# Expected next coordinate (Ground Truth):
actual_next_coordinate = np.array([6.0, 6.1], dtype=np.float32)

# Converting to PyTorch tensors and add batch dimension (batch_size=1):
input_tensor = torch.from_numpy(simulated_past_coordinates).unsqueeze(0)
target_tensor = torch.from_numpy(actual_next_coordinate).unsqueeze(0)

In [4]:
# Initializing Model, Loss Function, and Optimizer:
model = TrajectoryTrackerRNN(input_size=2, hidden_size=16, output_size=2)
criterion = nn.MSELoss() # Mean Squared Error is standard for coordinate regression
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [5]:
# Quick Training Loop (Overfitting on this single path for demonstration):
print('Training the tracking algorithm...')
for epoch in range(100):
    model.train()
    optimizer.zero_grad()

    # Forward pass:
    predictions = model(input_tensor)
    loss = criterion(predictions, target_tensor)

    # Backward pass and optimization:
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')

Training the tracking algorithm...
Epoch [20/100], Loss: 3.9541
Epoch [40/100], Loss: 0.0266
Epoch [60/100], Loss: 0.0644
Epoch [80/100], Loss: 0.0022
Epoch [100/100], Loss: 0.0006


In [6]:
# Testing the Prediction:
model.eval()
with torch.no_grad():
    predicted_coords = model(input_tensor).numpy()[0]

# Displaying the results:
print('\n--- Tracking Results ---')
print(f'Past History (Input):\n{simulated_past_coordinates}')
print(f'Predicted Next Position: [X: {predicted_coords[0]:.2f}, Y: {predicted_coords[1]:.2f}]')
print(f'Actual Next Position:    [X: {actual_next_coordinate[0]:.2f}, Y: {actual_next_coordinate[1]:.2f}]')


--- Tracking Results ---
Past History (Input):
[[1.  1.2]
 [2.  2.1]
 [3.1 2.9]
 [4.  4.2]
 [4.9 5. ]]
Predicted Next Position: [X: 6.00, Y: 6.07]
Actual Next Position:    [X: 6.00, Y: 6.10]
